# C.3 — LoRA trên Q/V và bộ nhớ QLoRA 4 bit
**Đề tài 9, tr. 19.** Chọn **Qwen/Qwen2.5-1.5B-Instruct** làm mô hình nền cho phép tính, không phải quyết định mô hình D.3. Đây là bản Instruct trước khi gắn adapter của nhóm. Chạy tuần tự bằng Python + PyTorch từ gốc repo hoặc thư mục notebooks; không cần GPU, mạng, trọng số hay dữ liệu QA.

**Nguồn đầu vào:** [config.json khóa theo revision 989aa7980e4cf806f80c7fef2b1adb7bc71aa306](https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct/blob/989aa7980e4cf806f80c7fef2b1adb7bc71aa306/config.json), [định nghĩa attention, MLP và RMSNorm của Qwen2 trong Transformers 4.43.1](https://github.com/huggingface/transformers/blob/v4.43.1/src/transformers/models/qwen2/modeling_qwen2.py). Đối chiếu tổng số phần tử với [metadata safetensors của revision](https://huggingface.co/api/models/Qwen/Qwen2.5-1.5B-Instruct/revision/989aa7980e4cf806f80c7fef2b1adb7bc71aa306).

## 1. Đếm tham số được huấn luyện
Có $L=28$ khối, chiều ẩn $d=1536$, 12 đầu Q nhưng chỉ 2 đầu K/V (grouped-query attention). Mỗi đầu có $1536/12=128$ chiều. Vì vậy trọng số Q có dạng **1536 × 1536**, còn V là **256 × 1536** (đầu ra × đầu vào). Việc lặp K/V để tính attention không tạo thêm trọng số.

Theo [LoRA, §4.1](../references/LoRA.pdf), với một ma trận $W$ kích thước $o × i$, hai ma trận học được có kích thước $r × i$ và $o × r$, tức $r(i+o)$ tham số. Đóng băng toàn bộ mô hình nền, kể cả bias; chỉ gắn adapter độc lập vào q_proj và v_proj ở mỗi khối, không học K, O, embedding hay đầu đọc. Hệ số α/r không làm thay đổi số tham số.

Toàn mô hình: $P_Q=28(1536+1536)r=86.016r$; $P_V=28(1536+256)r=50.176r$; **$P=136.192r$** (dấu chấm phân cách hàng nghìn).

In [ ]:
from pathlib import Path
import platform
import sys
import torch
from torch import nn

here = Path.cwd().resolve()
root = next(p for p in (here, *here.parents) if (p / 'src/lora.py').is_file())
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
from src.lora import LoRALinear

layers, d, heads, kv_heads, f, vocab = 28, 1536, 12, 2, 8960, 151936
head_dim = d // heads
kv = kv_heads * head_dim
ranks = (4, 8, 16)
print(f'Python {platform.python_version()} | PyTorch {torch.__version__} | không dùng CUDA')
print('rank          Q          V       tổng')
trainable = {}
for r in ranks:
    counts = []
    for out_features in (heads * head_dim, kv):
        # Một đại diện mỗi loại: numel trên meta không cấp vùng nhớ cho trọng số.
        adapter = LoRALinear(nn.Linear(d, out_features, bias=True, device='meta'), rank=r, alpha=r)
        measured = layers * sum(p.numel() for p in adapter.parameters() if p.requires_grad)
        assert measured == layers * r * (d + out_features)
        counts.append(measured)
    q_count, v_count = counts
    trainable[r] = q_count + v_count
    print(f'{r:4} {q_count:10,} {v_count:10,} {trainable[r]:10,}')

## 2. Ước lượng phần thường trú trên một GPU
**Kịch bản SFT, không offload, không reference DPO.** Phân biệt trọng số lưu NF4 với kiểu tính toán 16 bit. Bảy ma trận lớn trong mỗi khối (Q/K/V/O và gate/up/down) lưu NF4 với double quantization. Gọi $k=256$ là chiều ra K/V và $f=8960$ là chiều trung gian. Số phần tử lượng tử hóa là $N_4=L(2d^2+2dk+3df)$. Theo [QLoRA, §3, tr. 5](../references/QLora.pdf), mỗi phần tử dùng trung bình $4+8/64+32/(64·256)$ bit kể cả scale hai cấp. Bỏ qua codebook, offset và padding theo tensor.

Giả sử embedding/đầu đọc **chung** một trọng số (tie_word_embeddings=true), RMSNorm và bias đều lưu 16 bit. Số phần tử này là $N_{16}=V_{ocab}d+(2L+1)d+L(d+2k)$: một embedding, hai norm mỗi khối và một norm cuối, bias Q/K/V. Không đếm lm_head lần thứ hai. Đây là giả định lưu trữ, không phải mặc định của mọi thư viện: nếu chuyển phần này sang FP32 thì cộng thêm $2N_{16}$ byte.

Giả sử adapter lưu FP32 (autocast khi nhân), gradient FP32, hai moment AdamW FP32 thường trú: **16 byte/tham số học**; không cộng thêm bản master. Không tính gradient hay optimizer cho backbone đóng băng. Sau khi trạng thái optimizer đã cấp phát và gradient còn sống: $M_{tĩnh}=N_4 b_4/8+2N_{16}+16P$ byte. Dùng GiB = $2^{30}$ byte.

In [ ]:
nf4 = layers * (2*d*d + 2*d*kv + 3*d*f)
fp16 = vocab*d + (2*layers + 1)*d + layers*(d + 2*kv)
# Đối chứng độc lập: tổng BF16 của checkpoint trên Hub, trước lượng tử hóa.
assert nf4 + fp16 == 1_543_714_304
bits_nf4 = 4 + 8/64 + 32/(64*256)
gib = 2**30
base_bytes = nf4 * bits_nf4 / 8 + 2*fp16
print(f'{nf4:,} phần tử NF4; {fp16:,} phần tử 16 bit')
print(f'Nền đóng băng: {base_bytes/gib:.4f} GiB; nếu phần ngoài NF4 là FP32: +{2*fp16/gib:.4f} GiB')

## 3. Từ bộ nhớ tĩnh tới một kịch bản có activation
Chọn **microbatch $B=1$, chiều dài $T=512$, checkpoint từng khối, không KV cache**. Ước lượng các tensor chính ở pha tính loss: giữ một đầu vào FP16 cho mỗi khối ($2LBTd$ byte), cùng logits FP32 và một tensor FP32 cùng cỡ cho loss không fused ($2 × 4BTV_{ocab}$ byte). Cộng các khoản này vào ngân sách tĩnh để minh họa; đây **không** phải tổng thực đo, cận trên peak hay yêu cầu VRAM tối thiểu. Một implementation fused có thể tránh materialize hai tensor từ vựng; thời điểm sống của gradient cũng phụ thuộc lịch cập nhật.

Khi tăng $T$, hai khoản trên tăng tuyến tính; nếu attention lưu điểm dense, riêng một ma trận điểm FP16 kích thước $B × h × T × T$ tăng bậc hai. Tái tính activation/backward của block, buffer giải lượng tử, workspace GEMM/attention, CUDA context, optimizer tạm và phân mảnh allocator chưa được cộng. Vì vậy không suy từ bảng rằng một GPU cụ thể sẽ chạy vừa SFT hoặc DPO.

In [ ]:
batch, tokens = 1, 512
boundary_bytes = 2 * layers * batch * tokens * d
loss_bytes = 2 * 4 * batch * tokens * vocab
print(f'Checkpoint đầu khối: {boundary_bytes/gib:.4f} GiB; logits và loss: {loss_bytes/gib:.4f} GiB')
print('rank   adapter+grad+Adam (MiB)   tĩnh (GiB)    tĩnh + tensor đã nêu (GiB)')
for r, count in trainable.items():
    adapter_bytes = 16 * count
    static = base_bytes + adapter_bytes
    print(f'{r:4} {adapter_bytes/2**20:24.4f} {static/gib:12.4f} {(static+boundary_bytes+loss_bytes)/gib:29.4f}')

**Kết luận:** tăng rank từ 4 lên 16 làm số tham số học tăng 4 lần nhưng chỉ thêm khoảng 25 MiB trạng thái adapter, không nhân bốn bộ nhớ cả mô hình. Kiểm numel ở trên dùng [LoRA A.3](../src/lora.py) trên [tensor meta](https://docs.pytorch.org/docs/stable/meta.html) để kiểm kích thước, **không** lượng tử hóa hay tải Qwen. Muốn xác nhận VRAM cho D.3, phải đo riêng cấu hình thật qua ít nhất một bước forward, backward và optimizer sau khi cấp phát trạng thái; ghi cả peak allocated và peak reserved. Không cần chạy bước GPU đó để tái lập C.3.